In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()  # 이거 꼭 있어야됨

token = os.getenv("HF_TOKEN")
login(token)


In [2]:
import torch
import torch.nn as nn
from transformers import AutoModel
import inspect

model = AutoModel.from_pretrained(
    "facebook/dinov3-vits16plus-pretrain-lvd1689m"
)

print("=== model loaded ===")
print(model)


Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

=== model loaded ===
DINOv3ViTModel(
  (embeddings): DINOv3ViTEmbeddings(
    (patch_embeddings): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (rope_embeddings): DINOv3ViTRopePositionEmbedding()
  (layer): ModuleList(
    (0-11): 12 x DINOv3ViTLayer(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attention): DINOv3ViTAttention(
        (k_proj): Linear(in_features=384, out_features=384, bias=False)
        (v_proj): Linear(in_features=384, out_features=384, bias=True)
        (q_proj): Linear(in_features=384, out_features=384, bias=True)
        (o_proj): Linear(in_features=384, out_features=384, bias=True)
      )
      (layer_scale1): DINOv3ViTLayerScale()
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): DINOv3ViTGatedMLP(
        (gate_proj): Linear(in_features=384, out_features=1536, bias=True)
        (up_proj): Linear(in_features=384, out_features=1536, bias=True)
        (d

In [3]:
from datasets import load_dataset
from src.data import setdata
from torch.utils.data import DataLoader
data=load_dataset('zh-plus/tiny-imagenet')
train_data=setdata(data['train'])
test_data=setdata(data['valid'])
train_loader=DataLoader(train_data,batch_size=128,pin_memory=True,num_workers=4,shuffle=True)
test_loader=DataLoader(test_data,batch_size=128,pin_memory=True,num_workers=4,shuffle=True)

In [4]:
from src.re_make_Gate import inject_gating
from src.model import dinosplus_classfier
model, hooks = inject_gating(model, gate_type="elementwise")

vit=dinosplus_classfier(model,200)

In [5]:
for i in vit.parameters():
    i.required_grad=True

In [6]:
import torch.nn as nn
import torch.optim as optim
import torch
optimizer=optim.Adam(vit.parameters(),lr=0.0001)
criterion=nn.CrossEntropyLoss()
scaler=torch.amp.GradScaler()

In [7]:
import torch
print(torch.cuda.get_device_name(0))
device= 'cuda' if torch.cuda.is_available() else 'cpu'

NVIDIA GeForce RTX 5060 Ti


In [8]:
import torch

torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(False)

In [9]:
from src.training import train 
#train(vit,train_loader,test_loader,50,criterion,scaler,device,optimizer,'full fine tuning vit dino v3 splus_ gated retry','/home/hyuksu/deep-learning-study/outputs/best dino v3 splus gated retry.pth')

In [10]:
import torch
import torch.nn as nn
from transformers import AutoModel
import inspect

model2 = AutoModel.from_pretrained(
    "facebook/dinov3-vitb16-pretrain-lvd1689m"
)

print("=== model loaded ===")
print(model)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

=== model loaded ===
DINOv3ViTModel(
  (embeddings): DINOv3ViTEmbeddings(
    (patch_embeddings): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (rope_embeddings): DINOv3ViTRopePositionEmbedding()
  (layer): ModuleList(
    (0-11): 12 x DINOv3ViTLayer(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attention): DINOv3ViTAttention(
        (k_proj): Linear(in_features=384, out_features=384, bias=False)
        (v_proj): Linear(in_features=384, out_features=384, bias=True)
        (q_proj): Linear(in_features=384, out_features=384, bias=True)
        (o_proj): GatedOutputProjection(
          (original_o_proj): Linear(in_features=384, out_features=384, bias=True)
          (gate_proj): Linear(in_features=384, out_features=384, bias=True)
        )
      )
      (layer_scale1): DINOv3ViTLayerScale()
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): DINOv3ViTGatedMLP(
        (gate_proj):

In [11]:
from src.re_make_Gate import inject_gating
from src.model import dinosplus_classfier
model2, hooks = inject_gating(model2, gate_type="elementwise")

vit2=dinosplus_classfier(model2,200)

In [12]:
for i in vit2.parameters():
    i.requires_grad=True

In [13]:
import torch.optim as optim
import torch.nn as nn
optimizer2=optim.Adam(vit2.parameters(),lr=0.0001)
criterion2=nn.CrossEntropyLoss()

In [14]:
train_loader=DataLoader(train_data,batch_size=64,pin_memory=True,num_workers=4,shuffle=True,persistent_workers=True,prefetch_factor=4)
test_loader=DataLoader(test_data,batch_size=64,pin_memory=True,num_workers=4,shuffle=True,persistent_workers=True,prefetch_factor=4)

In [15]:
from src.training import train 
train(vit2,train_loader,test_loader,50,criterion2,scaler,device,optimizer2,'full fine tuning vit dino v3 B _gated retry','/home/hyuksu/deep-learning-study/outputs/best dino v3 B gated retry.pth')

  0%|          | 0/50 [00:00<?, ?it/s]

[Gate] mean=0.5049, min=0.4221, max=1.0000
[Gate] mean=0.5059, min=0.4163, max=1.0000
[Gate] mean=0.5015, min=0.4451, max=1.0000
[Gate] mean=0.4744, min=0.2125, max=1.0000
[Gate] mean=0.5020, min=0.4412, max=1.0000
[Gate] mean=0.5039, min=0.3989, max=1.0000
[Gate] mean=0.4744, min=0.1561, max=1.0000
[Gate] mean=0.5044, min=0.3662, max=1.0000
[Gate] mean=0.5083, min=0.3181, max=1.0000
[Gate] mean=0.5015, min=0.3909, max=1.0000
[Gate] mean=0.5020, min=0.3933, max=1.0000
[Gate] mean=0.5024, min=0.5000, max=1.0000
[Gate] mean=0.5039, min=0.3777, max=1.0000
[Gate] mean=0.4202, min=0.0421, max=1.0000
[Gate] mean=0.5039, min=0.3860, max=1.0000
[Gate] mean=0.5059, min=0.3376, max=1.0000
[Gate] mean=0.5146, min=0.1913, max=1.0000
[Gate] mean=0.5049, min=0.3943, max=1.0000
[Gate] mean=0.5039, min=0.3921, max=1.0000
[Gate] mean=0.5029, min=0.3918, max=1.0000
[Gate] mean=0.5049, min=0.4026, max=1.0000
[Gate] mean=0.5024, min=0.5000, max=1.0000
[Gate] mean=0.5054, min=0.3923, max=1.0000
[Gate] mean

2026/03/22 14:00:46 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


최고 성능 모델이 MLflow에 저장되었습니다.
